In [3]:
import numpy as np
import pickle
import src.rl
import joblib

/home/sikha/bse/rl/rl-graph-rag-retrieval/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Just double checking number of centroids matches number of different cluster labels

In [4]:
# 1. Load the document labels with allow_pickle=True
doc_labels = np.load(
    "../data/artifacts/clustering_output/doc_hierarchical_labels.npy", 
    allow_pickle=True
)

# Since it's an object array, let's check its shape and content to be safe:
print(f"Loaded doc_labels shape: {doc_labels.shape}")
print(f"First element looks like: {doc_labels[0]}")

# (If your array is 2D or contains lists/tuples, e.g., ['coarse_ID', 'fine_ID'], grab just the fine labels)
if doc_labels.ndim > 1:
    labels_fine = doc_labels[:, 1]  
elif isinstance(doc_labels[0], (list, tuple)) and len(doc_labels[0]) > 1:
    # In case it's a 1D array of lists/tuples
    labels_fine = np.array([item[1] for item in doc_labels])
else:
    labels_fine = doc_labels

# Load the dictionary of Fine KMeans models using joblib
kmeans_fine_dict = joblib.load("../data/artifacts/clustering_output/kmeans_fine_dict.pkl")

# Extract and stack the centroids
centroids_fine_list = []
for coarse_id in sorted(kmeans_fine_dict.keys()):
    fine_model = kmeans_fine_dict[coarse_id]
    centroids_fine_list.append(fine_model.cluster_centers_)

centroids_fine = np.vstack(centroids_fine_list)

# 4. Run the environment sanity check
print(f"Found {len(np.unique(labels_fine))} unique labels and {len(centroids_fine)} centroids.")
assert len(np.unique(labels_fine)) == len(centroids_fine), "Mismatch between clusters and centroids!"

Loaded doc_labels shape: (101962,)
First element looks like: C11_F14
Found 169 unique labels and 169 centroids.


In [5]:
%run ../src/rl.py --device cpu

Loading artifacts ...


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 1468.74it/s]


AttributeError: XLMRobertaTokenizer has no attribute prepare_for_model

In [6]:
import sys
import numpy as np
import pandas as pd

# 1. Ensure Python can find your src module
sys.path.append("..") 
from src.rl import load_artifacts, extract_top_k

# 2. Load the exact artifacts dictionary used by the RL environment
a = load_artifacts()

# 3. Filter for the validation questions (just like the evaluate function does)
qs = a["questions"]
val = qs[qs["split"] == "validation"]

# 4. Run the Oracle evaluation
recalls = []
for _, row in val.iterrows():
    # Make sure we only process queries that actually exist in the embeddings
    if row["question_id"] not in a["qid_to_qi"]:
        continue
        
    qi     = a["qid_to_qi"][row["question_id"]]
    q_norm = a["query_emb"][qi]
    
    # Calculate similarities and get top 4 clusters
    sims   = a["fine_cent"] @ q_norm
    top10   = np.argsort(sims)[::-1][:10]
    
    retrieved = set()
    for c in top4:
        # Extract top 5 docs from each cluster
        docs = extract_top_k(c, q_norm, a, k=5)
        retrieved.update(docs)
    
    # Calculate recall
    gold  = set(row["gold_node_ids"])
    if len(gold) > 0:
        hits  = len(retrieved & gold)
        recalls.append(hits / len(gold))

print(f"\nOracle Greedy Ceiling (Top 4 Clusters): {np.mean(recalls)*100:.1f}%")

# what's the max achievable recall at budget=20 for your current clustering?
for n_commits in [4, 6, 8, 10, 15, 20]:
    recalls = []
    for _, row in val.iterrows():
        qi    = a["qid_to_qi"][row["question_id"]]
        q_norm = a["query_emb"][qi]
        sims   = a["fine_cent"] @ q_norm
        top_c  = np.argsort(sims)[::-1][:n_commits]
        k_each = max(1, 20 // n_commits)
        retrieved = set()
        for c in top_c:
            retrieved.update(extract_top_k(c, q_norm, a, k=k_each))
        gold = set(row["gold_node_ids"])
        recalls.append(len(retrieved & gold) / len(gold))
    print(f"commits={n_commits:3d}  k_each={k_each}  ceiling={np.mean(recalls)*100:.1f}%")

Loading artifacts ...


NameError: name 'top4' is not defined

In [ ]:
import numpy as np

# 1. Load the hierarchical labels (e.g., array of "C11_F14")
raw_labels = np.load("../data/artifacts/clustering_output/doc_hierarchical_labels.npy", allow_pickle=True)

# 2. Extract the Coarse labels by splitting the string!
# If the label is "C11_F14", splitting by "_" and taking the first part gives "C11"
raw_coarse = np.array([lbl.split('_')[0] for lbl in raw_labels])

# 3. Convert string labels (e.g., 'C11') to integer IDs (0 to N-1)
unique_coarse = np.unique(raw_coarse)
coarse_to_id = {lbl: i for i, lbl in enumerate(unique_coarse)}

# 4. INJECT it directly into your existing 'a' dictionary
a["labels_coarse"] = np.array([coarse_to_id[lbl] for lbl in raw_coarse])

print(f"✅ Successfully populated a['labels_coarse'] with {len(unique_coarse)} unique coarse clusters.")


# 3. Convert string labels (e.g., 'C11') to integer IDs (0 to N-1)
unique_coarse = np.unique(raw_coarse)
coarse_to_id = {lbl: i for i, lbl in enumerate(unique_coarse)}

# 4. INJECT it directly into your existing 'a' dictionary!
a["labels_coarse"] = np.array([coarse_to_id[lbl] for lbl in raw_coarse])

print(f"✅ Successfully populated a['labels_coarse'] with {len(unique_coarse)} unique coarse clusters.")

# quick ceiling check at different n_fine values
# without reclustering — simulate by using coarse clusters directly
for n_coarse in [10, 20, 30, 40]:
    # treat each coarse cluster as one "pick"
    recalls = []
    for _, row in val.iterrows():
        qi     = a["qid_to_qi"][row["question_id"]]
        q_norm = a["query_emb"][qi]
        # compute sim to coarse centroids
        coarse_cent = normalize(
            np.array([a["doc_emb"][a["labels_coarse"]==c].mean(axis=0) 
                      for c in range(n_coarse)]), norm="l2")
        sims   = coarse_cent @ q_norm
        top4   = np.argsort(sims)[::-1][:4]
        retrieved = set()
        for c in top4:
            members = np.where(a["labels_coarse"]==c)[0]
            doc_sims = a["doc_emb"][members] @ q_norm
            top5 = members[np.argsort(doc_sims)[::-1][:5]]
            retrieved.update(top5.tolist())
        gold = set(row["gold_node_ids"])
        recalls.append(len(retrieved & gold) / len(gold))
    print(f"n_coarse={n_coarse:3d}  ceiling={np.mean(recalls)*100:.1f}%")

✅ Successfully populated a['labels_coarse'] with 20 unique coarse clusters.
✅ Successfully populated a['labels_coarse'] with 20 unique coarse clusters.
n_coarse= 10  ceiling=26.1%
n_coarse= 20  ceiling=44.7%


/tmp/ipykernel_141686/4028146222.py:39: RuntimeWarning: Mean of empty slice
  np.array([a["doc_emb"][a["labels_coarse"]==c].mean(axis=0)
/home/sikha/bse/rl/rl-graph-rag-retrieval/.venv/lib/python3.13/site-packages/numpy/_core/_methods.py:134: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(


ValueError: Input contains NaN.

In [ ]:
# For each question, what rank is the gold doc's cluster?
ranks = []
for _, row in val.iterrows():
    qi     = a["qid_to_qi"][row["question_id"]]
    q_norm = a["query_emb"][qi]
    sims   = a["fine_cent"] @ q_norm
    cluster_rank = np.argsort(sims)[::-1]
    
    for gold_node in row["gold_node_ids"]:
        gold_cluster = a["labels_fine"][gold_node]
        rank = np.where(cluster_rank == gold_cluster)[0][0]
        ranks.append(rank)

ranks = np.array(ranks)
print(f"Gold cluster rank=0  : {(ranks==0).mean()*100:.1f}%")
print(f"Gold cluster rank<=1 : {(ranks<=1).mean()*100:.1f}%")
print(f"Gold cluster rank<=3 : {(ranks<=3).mean()*100:.1f}%")
print(f"Gold cluster rank<=9 : {(ranks<=9).mean()*100:.1f}%")
print(f"Gold cluster rank<=19: {(ranks<=19).mean()*100:.1f}%")
print(f"Mean rank: {ranks.mean():.1f}")
print(f"Median rank: {np.median(ranks):.1f}")

Gold cluster rank=0  : 31.9%
Gold cluster rank<=1 : 45.8%
Gold cluster rank<=3 : 59.2%
Gold cluster rank<=9 : 76.2%
Gold cluster rank<=19: 85.0%
Mean rank: 9.8
Median rank: 2.0
